## Projet 2 : Audio — "L'Oreille Intelligente"

Nous allons maintenant créer une interface capable d'écouter l'environnement et de reconnaître les sons urbains du dataset **UrbanSound8K**.

### Objectifs :

1. **Interface de contrôle :** Créez une interface simple (ou utilisez des touches clavier comme 's' pour Start et 'q' pour Quit) pour lancer l'écoute.
2. **Gestion du Buffer :** Le modèle a été entraîné sur des extraits de 4 secondes. Vous devez créer un **buffer audio** (un tampon) qui accumule les données du micro avant de les envoyer au modèle.
3. **Prétraitement "On-the-fly" :** À chaque intervalle (ex: toutes les 2 secondes), extrayez les caractéristiques (Mel-Spectrogramme ou MFCC) du buffer actuel.
4. **Classification continue :** Affichez à l'écran la classe détectée (ex: "Siren", "Street Music", "Drilling") en mettant à jour le texte dès qu'un nouveau son est identifié.

### Contraintes techniques :

* **Fréquence d'échantillonnage :** Assurez-vous que le flux micro correspond au `sr` utilisé lors de l'entraînement (ex: 16kHz ou 22.050Hz).
* **Glissement (Overlap) :** Pour plus de réactivité, essayez de faire des prédictions glissantes (ex: prédire toutes les secondes sur les 4 dernières secondes écoulées).


In [2]:
from keras.models import load_model

try:
    model = load_model('model_urbansound8k.keras')
    print("Modèle chargé avec succès !")
    model.summary()
except Exception as e:
    print(f"Erreur lors du chargement : {e}")

Modèle chargé avec succès !


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ mel_input           │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 128, 128,  │        320 │ mel_input[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        128 │ conv2d_6[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mfcc_input          │ (None, 240)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_6     │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 512)       │    123,392 │ mfcc_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 64, 64,    │     36,992 │ max_pooling2d_6[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512)       │      2,048 │ dense_10[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        512 │ conv2d_7[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 512)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_7     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 256)       │    131,328 │ dropout_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 32, 32,    │    590,336 │ max_pooling2d_7[… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_11[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │      2,048 │ conv2d_8[0][0]    │
│ (BatchNormalizatio… │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_8     │ (None, 16, 16,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 128)       │     32,896 │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 3,353,184 (12.79 MB)

 Trainable params: 1,116,682 (4.26 MB)

 Non-trainable params: 3,136 (12.25 KB)

 Optimizer params: 2,233,366 (8.52 MB)

In [3]:
import cv2
import numpy as np
import pyaudio
import sys
import tensorflow as tf
import librosa

# --- CONFIGURATION PARAMÉTRÉ ---

# Audio
SR = 16000            # Fréquence d'échantillonnage
CHUNK = 1024          # Taille du buffer (plus petit = moins de latence, plus de CPU)
FORMAT = pyaudio.paFloat32
CHANNELS = 1          # Mono
DURATION = 1.0        # Durée de l'écoute en secondes
RATE = int(SR * DURATION)  # Nombre d'échantillons à lire par buffer
TOP_DB = 30


NUM_MEL_BINS = 128
NUM_SPECTROGRAM_BINS = 257

# On crée la matrice une fois pour toutes
MEL_WEIGHT_MATRIX = tf.signal.linear_to_mel_weight_matrix(
    NUM_MEL_BINS, 
    NUM_SPECTROGRAM_BINS, 
    SR, 
    80.0, 
    7600.0
)


# --- INITIALISATION ---

# Initialisation de l'audio


def preprocess_audio(audio_data):
    """Preprocess the audio data to match the model's input shape."""
    # -----------------------------------
    # 2. Traitement MEL_SPECTROGRAM
    # -----------------------------------
    # on recupère le signal et on le transforme en Tenseur
    audio_tf = tf.convert_to_tensor(audio_data, dtype=tf.float32)

    # on fait le spectrogramme de la transformée de fournier
    stft = tf.signal.stft(audio_tf, frame_length=512, frame_step=256, fft_length=512)
    spectrogram = tf.abs(stft)
    
    # 3. Conversion Mel-Scale (en TensorFlow !)
   
    mel_spectrogram = tf.tensordot(spectrogram, MEL_WEIGHT_MATRIX, 1)
    mel_spectrogram.set_shape(spectrogram.shape[:-1].concatenate(
      MEL_WEIGHT_MATRIX.shape[-1:]))

    # 4. Log-Scale (Décibels)
    log_mel_spec = tf.math.log(mel_spectrogram + 1e-6)

    # 5. Normalisation et Resize
    spec = tf.expand_dims(log_mel_spec, axis=-1)
    spec = tf.ensure_shape(spec, [None, NUM_MEL_BINS, 1])
    
    # 4. Resize & Log
    spec = tf.image.resize(spec, [128, 128])
    mean = tf.reduce_mean(spec)
    std = tf.math.reduce_std(spec)
    input_mel = (spec - mean) / (std + 1e-6)

    # -----------------------------------
    # 2. Traitement MFCC
    # -----------------------------------
    intervals = librosa.effects.split(audio_data, top_db=TOP_DB)
    
    # 2. Concatenate only the non-silent chunks
    if len(intervals) > 0 : 
        audio_active = np.concatenate([audio_tf[start:end] for start, end in intervals])
    else : 
        audio_active = audio_tf[:SR]
    # On limite/pad à 1 seconde exactement (16000 samples)
    
    if len(audio_active) >= SR:
        audio_active = audio_active[:SR]
    else:
        # FIX IS HERE: use len(audio_active)
        padding_needed = SR - len(audio_active)
        audio_active = np.pad(audio_active, (0, padding_needed))

    audio_tf_MFCC = tf.convert_to_tensor(audio_active, dtype=tf.float32)
    stft_MFCC = tf.signal.stft(audio_tf_MFCC, frame_length=512, frame_step=256, fft_length=512)
    spectrogram_MFCC = tf.abs(stft_MFCC)
    
    # 3. Conversion Mel-Scale (en TensorFlow !)
   
    mel_spectrogram_MFCC = tf.tensordot(spectrogram_MFCC, MEL_WEIGHT_MATRIX, 1)
    mel_spectrogram_MFCC.set_shape(spectrogram_MFCC.shape[:-1].concatenate(
      MEL_WEIGHT_MATRIX.shape[-1:]))

    # 4. Log-Scale (Décibels)
    log_mel_MFCC = tf.math.log(mel_spectrogram_MFCC + 1e-6)

    mfccs = tf.signal.mfccs_from_log_mel_spectrograms(log_mel_MFCC)[..., :40]
    delta = mfccs[1:, :] - mfccs[:-1, :]
    delta2 = delta[1:, :] - delta[:-1, :]

    def get_stats(tensor):
            m = tf.reduce_mean(tensor, axis=0)
            s = tf.math.reduce_std(tensor, axis=0)
            return m, s

    m_mfcc, s_mfcc = get_stats(mfccs)
    m_delta, s_delta = get_stats(delta)
    m_delta2, s_delta2 = get_stats(delta2)

    # 5. Concatenation finale : [40]*6 = 240
    mfcc_vector = tf.concat([m_mfcc, s_mfcc, m_delta, s_delta, m_delta2, s_delta2], axis=0)
    mfcc_vector.set_shape([240])

    return {"mel_input":tf.expand_dims(input_mel, 0), "mfcc_input":tf.expand_dims(mfcc_vector, 0)}
    # return {"mel_input":input_mel, "mfcc_input":mfcc_vector}



In [4]:
def record_and_predict():
    
    p = pyaudio.PyAudio()
    print("Système prêt. Appuyez sur 'q' pour quitter.")

    try:
        stream = p.open(format=FORMAT, channels=CHANNELS, rate=SR, 
                        input=True, frames_per_buffer=CHUNK)
    except Exception as e:
        print(f"Erreur Audio : {e}")
        sys.exit()

    frames = []
    num_chunks = int(SR / CHUNK)

    for _ in range(0, num_chunks):
        data = stream.read(CHUNK, exception_on_overflow=False)
        if data :
            frames.append(np.frombuffer(data, dtype=np.float32))
       
    stream.stop_stream()
    stream.close()
    p.terminate()
    print("Echantillon audio capturé.")

    if len(frames) == 0:
        print("❌ Erreur : Aucun échantillon audio capturé.")
        return

    audio_np = np.concatenate(frames)

    inputs = preprocess_audio(audio_np)

    preds = model.predict(inputs, verbose=0)

    class_id = np.argmax(preds)

    dict_class_id = {
        3:'dog_bark',
        2:'children_playing',
        1:'car_horn',
        0:'air_conditioner',
        9:'street_music',
        8:'siren',
        5:'engine_idling',
        7:'jackhammer',
        4:'drilling',
        6:'gun_shot',
    }

    nom_class = dict_class_id.get(class_id, "Inconnu")

    prob = np.max(preds)

    print(f"Résultat : Classe {nom_class} ({prob*100:.1f}%)")

    if nom_class == "dog_bark":
        return True
    else:
        return False

    

In [5]:
def sentinel_img ():    
    # Vidéo
    CAM_ID = 0          # Index de la caméra (0 = défaut)
    WIDTH = 640         # Largeur de capture
    HEIGHT = 480        # Hauteur de capture

    # --- INITIALISATION ---
    # Initialisation de la caméra
    cap = cv2.VideoCapture(CAM_ID)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, HEIGHT)

    ret, frame = cap.read()

    if ret:
        # Enregistre l'image sous le nom 'capture_audio.jpg'
        cv2.imwrite("capture_audio.jpg", frame)
        print("Image enregistrée avec succès !")
    else:
        print("Erreur : Impossible de lire le flux de la caméra.")
    
    cap.release()
    return frame

In [ ]:
import keyboard

print("\n--- INTERFACE DE CONTRÔLE ---")
print("Appuyez sur 'S' pour démarrer l'écoute (1s)")
print("Appuyez sur 'Q' pour quitter")

while True:
    dog_bark = record_and_predict()
    if dog_bark:
        sentinel_img()
        print("Dog bark detected!")
        
    if keyboard.is_pressed('q'):
        print("Fermeture...")
        break




--- INTERFACE DE CONTRÔLE ---
Appuyez sur 'S' pour démarrer l'écoute (1s)
Appuyez sur 'Q' pour quitter
Système prêt. Appuyez sur 'q' pour quitter.
Echantillon audio capturé.
Résultat : Classe engine_idling (97.8%)
Système prêt. Appuyez sur 'q' pour quitter.
Echantillon audio capturé.
Résultat : Classe engine_idling (96.4%)
Système prêt. Appuyez sur 'q' pour quitter.
Echantillon audio capturé.
Résultat : Classe dog_bark (95.5%)
Image enregistrée avec succès !
Dog bark detected!
Système prêt. Appuyez sur 'q' pour quitter.
Echantillon audio capturé.
Résultat : Classe dog_bark (98.1%)
Image enregistrée avec succès !
Dog bark detected!
Système prêt. Appuyez sur 'q' pour quitter.
Echantillon audio capturé.
Résultat : Classe engine_idling (98.1%)
Système prêt. Appuyez sur 'q' pour quitter.
Echantillon audio capturé.
Résultat : Classe engine_idling (95.4%)
Système prêt. Appuyez sur 'q' pour quitter.
Echantillon audio capturé.
Résultat : Classe dog_bark (92.2%)
Image enregistrée avec succès !


NameError: name 'p' is not defined